# Kim et al. (2020): removal operations in working memory -- item-level RSA post hoc

Study-specific analyses of the item-level ("weighted") RSA of Kim, Smolker, Smith, Banich & Lewis-Peacock (2020), *Nat Commun* 11:6239, run on the core pipeline's output. The core pipeline (`workflows/run_pipeline.py --config examples/config-kim2020-ncomms.example.json`) builds the feature-weighted item templates from the localizer and scores every study-phase TR against them. This notebook starts from its group raw tables (`<output_dir>/group/template_rsa/raw/`) and does what is specific to this design:

1. **Timecourses** of target / related / others similarity (and the *replacement category* on replace trials), per operation.
2. **Removal from WM** (Fig. 4b, item level): each operation minus maintain, baseline-corrected, tested in 3-TR bins from operation onset; removal onset = first bin reliably below maintain; between-operation comparisons per bin.
3. **Replacement tracking**: on replace trials, similarity to the templates of the category participants switched to, vs. the item being replaced.
4. **Encoding fidelity** (Fig. 5a): target > related > others for the presented item.
5. **Proactive interference** (Fig. 5b): encoding fidelity of item N+1 when it shares item N's category, by the operation performed on item N.

Each section draws its figures first, then its statistics. Formal definitions: `docs/lab_methods.md` (section K).

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# run from the repository root, so the relative paths in the configs resolve as in the pipeline
REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "_interactive_notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, REPO)

import _interactive_notebooks.rsa_posthoc_common as common
from _interactive_notebooks.rsa_posthoc_common import (
    add_previous_trial_columns, baseline_correct, bin_means, compare_groups_by_bin, difference_scores,
    first_significant_bin, join_item_attributes, load_config, load_master, load_scores, make_bins,
    make_toy_scores, mixed_model, one_sample_by_bin, paired_contrast, pairwise_tests, plot_points,
    plot_timecourse, rm_anova, save_outputs, subject_means, subtract_reference, summarize)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Configuration -- edit these

In [ ]:
# -- which core run and which slice of it --
CONFIG_PATH = os.environ.get("RSA_CONFIG", "examples/config-kim2020-ncomms.example.json")
ROI = "vvs"                       # a key of the config's `rois`
LEVEL = "item"                    # feature-weighting level: "item" (the paper), "category" or "none"
TIMECOURSE_SET = "study_timecourse"   # target-set names in template_rsa.targets
FIDELITY_SET = "encoding_fidelity"

# -- design columns (events.tsv) --
CONDITION_COL = "condition"
REFERENCE = "maintain"                          # removal = operation minus maintain
EXCLUDE_CONDITIONS = ["replace_subcategory"]    # the paper reports replace-category trials only
REPLACE_CONDITION = "replace_category"
REPLACE_CATEGORY_COL = "replace_category"       # the category switched to (the core's `replacement` set)
CATEGORY_COL = "category"

# -- timing, in seconds (converted with the data's TR, so the same values work at any TR) --
BASELINE_SECONDS = (0.0, 2.76)      # subtracted from each curve: the stimulus period (TRs 1-6 at 0.46 s)
STAT_WINDOW_SECONDS = (2.76, 9.66)  # tested: operation onset to the end of the longest fixation (TRs 7-21)
BIN_SECONDS = 1.38                  # bin width (3 TRs)

# -- statistics --
ALPHA = 0.05
FDR_METHOD = "fdr_bh"
N_BOOTSTRAP = 1000
SEED = 0

OUTPUT_DIR = os.environ.get("RSA_POSTHOC_OUT")  # None: <output_dir>/posthoc/kim2020/roi-<ROI>_level-<LEVEL>
common.configure(globals())

## Load the group raw table

One row per subject x trial x TR x comparison set; every events.tsv column rides along. Trials of `EXCLUDE_CONDITIONS` are dropped here, after nothing has been computed from them.

In [ ]:
cfg = load_config(CONFIG_PATH)
OUTPUT_DIR = OUTPUT_DIR or os.path.join(cfg["pipeline"]["output_dir"], "posthoc", "kim2020", f"roi-{ROI}_level-{LEVEL}")

tc = load_scores(cfg, ROI, TIMECOURSE_SET, LEVEL)
tc = tc[~tc[CONDITION_COL].isin(EXCLUDE_CONDITIONS)].reset_index(drop=True)

TR = float(np.median(np.diff(np.sort(tc["time_s"].unique()))))
to_tr = lambda seconds: int(round(seconds / TR))
BASELINE_TRS = (to_tr(BASELINE_SECONDS[0]), to_tr(BASELINE_SECONDS[1]))
FIRST_TR, LAST_TR = to_tr(STAT_WINDOW_SECONDS[0]), to_tr(STAT_WINDOW_SECONDS[1]) - 1
BINS = make_bins(FIRST_TR, LAST_TR, max(1, to_tr(BIN_SECONDS)))
STAT_SPAN = (FIRST_TR * TR, (LAST_TR + 1) * TR)
OPERATIONS = [REFERENCE] + sorted(set(tc[CONDITION_COL].dropna()) - {REFERENCE})
COLORS = {op: f"C{i}" for i, op in enumerate(OPERATIONS)}

print(f"{tc['subject'].nunique()} subjects, {tc.drop_duplicates(['subject', 'run_id', 'trial_index']).shape[0]} trials, "
      f"TR = {TR:g} s, {tc['tr_offset'].max() + 1} TRs per trial")
print(f"baseline TRs {BASELINE_TRS[0]}-{BASELINE_TRS[1] - 1}; tested TRs {FIRST_TR}-{LAST_TR} in bins {[b for b in BINS]}")
print("trials per operation:", tc.drop_duplicates(['subject', 'run_id', 'trial_index'])[CONDITION_COL].value_counts().to_dict())
figures, tables = {}, {}

## 1. Timecourses by operation

Trials are averaged within subject first; bands are mean +- 1 SEM across subjects. `replacement` exists only on replace trials (similarity to the templates of the category switched to).

In [ ]:
sub = subject_means(tc, [CONDITION_COL, "comparison", "tr_offset", "time_s"])
comparisons = [c for c in ("target", "related", "others", "replacement") if c in set(sub["comparison"])]

fig, axes = plt.subplots(1, len(comparisons), figsize=(4.3 * len(comparisons), 3.8), sharey=True, squeeze=False)
for ax, comp in zip(axes.flat, comparisons):
    plot_timecourse(ax, sub[sub["comparison"] == comp], CONDITION_COL, comp, colors=COLORS, shade=STAT_SPAN)
fig.suptitle(f"{ROI}, {LEVEL}-weighted templates: similarity from trial onset (raw)", fontsize=11)
fig.tight_layout()
figures["1_raw_timecourses"] = fig
plt.show()

## 2. Removal: each operation minus maintain (Fig. 4b, item level)

Each subject x operation curve is baseline-corrected (its mean over `BASELINE_SECONDS` subtracted), then maintain's curve is subtracted at every TR, so **y = 0 is maintain**. Below 0 = less of the item than when maintaining it. Per bin: a one-sample t-test of each operation vs 0 (the removal onset is the first bin reliably below 0, as in the paper; `p_fdr` corrects across the bins of each operation), and a repeated-measures ANOVA plus FDR-corrected pairwise tests across operations.

In [ ]:
bc = baseline_correct(sub, BASELINE_TRS, [CONDITION_COL, "comparison"])
removal = subtract_reference(bc[bc["comparison"] == "target"], CONDITION_COL, REFERENCE, ["tr_offset", "time_s"])
removal_binned = bin_means(removal, BINS, [CONDITION_COL])
removal_tests = one_sample_by_bin(removal_binned, [CONDITION_COL])
removal_onsets = first_significant_bin(removal_tests, [CONDITION_COL], direction="below", p_col="p")
between_operations = compare_groups_by_bin(removal_binned, CONDITION_COL)

fig, ax = plt.subplots(figsize=(7, 4.2))
plot_timecourse(ax, removal, CONDITION_COL, f"target similarity, operation minus {REFERENCE} (baseline-corrected)",
                tests=removal_tests, tr_seconds=TR, p_col="p", colors=COLORS, shade=STAT_SPAN)
fig.tight_layout()
figures["2_removal_minus_maintain"] = fig
plt.show()

print("Removal onset (first bin below maintain, p < ALPHA):")
display(removal_onsets[[CONDITION_COL, "bin", "mean", "t", "p", "p_fdr"]])
display(removal_tests)
display(between_operations)
tables.update(removal_vs_maintain_by_bin=removal_tests, removal_onsets=removal_onsets,
              removal_between_operations=between_operations)

## 3. Replacement tracking (replace trials)

On replace trials, the item being replaced (`target`) should fall while the category switched to (`replacement`: mean similarity to every template of that category) rises. Both curves are baseline-corrected; per bin, a one-sample t-test vs 0 for each, and a paired test of replacement minus target (the crossover).

In [ ]:
replace = bc[(bc[CONDITION_COL] == REPLACE_CONDITION) & bc["comparison"].isin(["target", "replacement"])]
if replace.empty:
    print(f"no {REPLACE_CONDITION} trials with a replacement score -- check REPLACE_CONDITION / the core extra_sets")
else:
    replace_binned = bin_means(replace, BINS, ["comparison"])
    replace_tests = one_sample_by_bin(replace_binned, ["comparison"])
    replacement_onset = first_significant_bin(replace_tests[replace_tests["comparison"] == "replacement"],
                                              ["comparison"], direction="above", p_col="p")
    crossover = paired_contrast(replace_binned, "comparison", "replacement", "target", by=["bin_start", "bin"])

    fig, ax = plt.subplots(figsize=(7, 4.2))
    plot_timecourse(ax, replace, "comparison", f"{REPLACE_CONDITION} trials: replaced item vs. replacement category",
                    tests=replace_tests, tr_seconds=TR, p_col="p", shade=STAT_SPAN)
    fig.tight_layout()
    figures["3_replacement_tracking"] = fig
    plt.show()
    print("Replacement onset (first bin above baseline):")
    display(replacement_onset[["bin", "mean", "t", "p"]])
    display(crossover)
    tables.update(replacement_by_bin=replace_tests, replacement_onset=replacement_onset, replacement_minus_target=crossover)

## 4. Encoding fidelity (Fig. 5a)

Similarity of the image-presentation pattern (window mean, lag-shifted) to the presented item's template (`target`), to the other items of its category (`related`) and to the items of other categories (`others`). The paper used a one-way repeated-measures ANOVA with Tukey-Kramer post hocs; here the post hocs are FDR-corrected paired t-tests.

In [ ]:
ef_all = load_scores(cfg, ROI, FIDELITY_SET, LEVEL)
ef = ef_all[~ef_all[CONDITION_COL].isin(EXCLUDE_CONDITIONS)]
ef_sub = subject_means(ef, ["comparison"])
F, p, n = rm_anova(ef_sub, "z", "comparison")
fidelity_pairs = pairwise_tests(ef_sub, "z", "comparison", levels=["target", "related", "others"])
fidelity_anova = pd.DataFrame([{"test": "rm_anova target/related/others", "F": F, "p": p, "n": n}])

fig, ax = plt.subplots(figsize=(4.5, 3.8))
plot_points(ax, ef_sub.assign(all="all trials"), "all", "comparison", "encoding fidelity",
            hue_order=["target", "related", "others"])
fig.tight_layout()
figures["4_encoding_fidelity"] = fig
plt.show()
display(fidelity_anova)
display(fidelity_pairs)
tables.update(encoding_fidelity_anova=fidelity_anova, encoding_fidelity_pairwise=fidelity_pairs)

## 5. Proactive interference: encoding fidelity of item N+1 (Fig. 5b)

Previous-trial columns are computed on the **full** trial sequence (before any exclusion), so "the previous trial" is always the trial that was actually shown. Then, as in the paper:
- trials whose previous trial was excluded (`EXCLUDE_CONDITIONS`) are dropped,
- replace trials whose replacement category equals item N+1's category are dropped (that overlap is a different effect),
- same − different category encoding fidelity is computed per subject and per operation on trial N; below 0 = proactive interference.

Tests: one-sample t of same − different per operation (with a subject-bootstrap CI), a repeated-measures ANOVA across operations, and FDR-corrected pairwise comparisons.

In [ ]:
prev_cond, same_cat = f"prev_{CONDITION_COL}", f"same_{CATEGORY_COL}"
seq = add_previous_trial_columns(ef_all[ef_all["comparison"] == "target"], [CATEGORY_COL, CONDITION_COL],
                                 matches={"same_replacement": (REPLACE_CATEGORY_COL, CATEGORY_COL)})
keep = (seq[same_cat].notna()
        & ~seq[CONDITION_COL].isin(EXCLUDE_CONDITIONS) & ~seq[prev_cond].isin(EXCLUDE_CONDITIONS)
        & ~((seq[prev_cond] == REPLACE_CONDITION) & (seq["same_replacement"] == "same")))
seq = seq[keep]
print("N+1 trials per previous operation x same/different:")
display(seq.groupby([prev_cond, same_cat]).size().unstack())

pi_sub = subject_means(seq, [prev_cond, same_cat])
interference = paired_contrast(pi_sub, same_cat, "same", "different", by=[prev_cond])
pi_diff = difference_scores(pi_sub, same_cat, "same", "different", by=[prev_cond])
F, p, n = rm_anova(pi_diff, "diff", prev_cond)
interference_anova = pd.DataFrame([{"test": f"rm_anova of same-different across {prev_cond}", "F": F, "p": p, "n": n}])
interference_pairs = pairwise_tests(pi_diff, "diff", prev_cond, levels=[o for o in OPERATIONS if o in set(pi_diff[prev_cond])])

fig, ax = plt.subplots(figsize=(5.5, 3.8))
plot_points(ax, pi_diff.assign(measure="same - different"), prev_cond, "measure",
            "encoding fidelity of item N+1: same - different category", value="diff",
            order=[o for o in OPERATIONS if o in set(pi_diff[prev_cond])])
ax.set_xlabel("operation on trial N")
fig.tight_layout()
figures["5_proactive_interference"] = fig
plt.show()
display(interference)
display(interference_anova)
display(interference_pairs)
tables.update(proactive_interference=interference, proactive_interference_anova=interference_anova,
              proactive_interference_pairwise=interference_pairs)

## Save outputs

`stats_and_figures.pdf` (every figure above, in order), one CSV per table, and `run_settings.json` with the settings of this run, in `OUTPUT_DIR`.

In [ ]:
run_settings = {k: globals()[k] for k in (
    "CONFIG_PATH", "ROI", "LEVEL", "TIMECOURSE_SET", "FIDELITY_SET", "CONDITION_COL", "REFERENCE", "EXCLUDE_CONDITIONS",
    "REPLACE_CONDITION", "REPLACE_CATEGORY_COL", "CATEGORY_COL", "BASELINE_SECONDS", "STAT_WINDOW_SECONDS", "BIN_SECONDS",
    "ALPHA", "FDR_METHOD", "N_BOOTSTRAP", "SEED", "TR", "BASELINE_TRS", "FIRST_TR", "LAST_TR", "BINS")}
for path in save_outputs(OUTPUT_DIR, figures, tables, run_settings):
    print(path)

## Methods

**Scores (core pipeline).** Templates: each item's z-scored localizer pattern (3 presentation TRs, shifted 4.6 s), averaged over its 5 repetitions, restricted to its category's selective voxels (category-vs-others t-contrast, p < .05, ≥ 10-voxel clusters, in VVS) and weighted by the item-vs-other-53-items t-map (least-squares-all GLM). Each study TR is weighted by the same item's t-map and correlated with the template; Pearson r is averaged over the items of a set and Fisher-z transformed. Sets: `target` (the trial's item), `related` (other items of its category), `others` (items of other categories), `replacement` (every template of the category switched to, replace trials only).

**Timecourses.** Trials are averaged within subject (subjects are the random effect). Curves are baseline-corrected by their mean over the stimulus period; removal curves subtract maintain's curve at each TR. Bins of `BIN_SECONDS` over `STAT_WINDOW_SECONDS` are tested with one-sample t-tests vs 0 (removal onset = first bin with p < ALPHA, mean < 0; `p_fdr` across bins per operation), repeated-measures ANOVAs across operations and FDR-corrected paired t-tests between operations.

**Encoding fidelity and proactive interference.** Window-mean similarity of the presentation pattern (shifted 4.6 s). Proactive interference: same − different category encoding fidelity of item N+1 by the operation on trial N, after excluding trials following excluded conditions and replace trials whose replacement category matched item N+1; one-sample t-tests with subject-bootstrap CIs, repeated-measures ANOVA across operations, FDR-corrected pairwise tests.

**Differences from the paper.** fMRIPrep instead of FSL preprocessing; nilearn GLMs instead of SPM; Tukey-Kramer post hocs replaced by FDR-corrected paired t-tests; the trial-balanced bootstrap (15 trials, 1000 iterations) is not reproduced -- see `docs/lab_methods.md`.

## Worked example (toy data)

A toy table with the same structure: 12 subjects, maintain and suppress, 12 TRs, and suppress losing 0.05 and then 0.12 of target similarity in the last two 3-TR bins. Subtracting maintain and testing by bin should leave the first two bins near 0 and the last two below 0.

In [ ]:
toy = make_toy_scores()
toy_sub = subject_means(toy, ["condition", "tr_offset", "time_s"])
toy_removal = subtract_reference(toy_sub, "condition", "maintain", ["tr_offset", "time_s"])
toy_tests = one_sample_by_bin(bin_means(toy_removal, make_bins(0, 11, 3), ["condition"]), ["condition"])
display(toy_tests[["condition", "bin", "n", "mean", "t", "p", "p_fdr"]])